In [1]:
import pandas as pd
import sys
sys.path.append('..')

from data.emission_factors import SCOPE1, SCOPE2_ELECTRICITY, BIOMETHANE
from data.sites import LACQ, FRENCH_GAS_SECTOR_AVG

site = LACQ
print(f"Building trajectory for: {site['name']}")

Building trajectory for: Lacq Gas Processing Site (illustrative)


In [3]:
def scenario_ppa(site, ef_electricity, ppa_fraction):
    assert 0.0 <= ppa_fraction <= 1.0, 'ppa_fraction must be between 0 and 1'
    grid = site['grid']
    mwh = site['electricity_MWh']
    ef_grid = ef_electricity[grid]
    ef_ppa  = ef_electricity['renewable_ppa']
    ef_new  = (ppa_fraction * ef_ppa) + ((1 - ppa_fraction) * ef_grid)
    scope2_baseline = mwh * ef_grid
    scope2_new      = mwh * ef_new
    return {
        'scope2_new_tCO2': round(scope2_new, 1),
        'reduction_tCO2':  round(scope2_baseline - scope2_new, 1),
    }

def scenario_biomethane(site, ef_scope1, ef_biomethane,
                        bio_fraction, feedstock='agricultural_waste'):
    assert 0.0 <= bio_fraction <= 1.0, 'bio_fraction must be between 0 and 1'
    ng_mwh       = site.get('natural_gas_MWh', 0)
    ng_remaining = ng_mwh * (1 - bio_fraction)
    bio_mwh      = ng_mwh * bio_fraction
    scope1_ng_new = ng_remaining * ef_scope1['natural_gas']
    scope1_bio    = bio_mwh * ef_biomethane[feedstock]
    scope1_oil    = site.get('fuel_oil_MWh', 0) * ef_scope1['fuel_oil']
    scope1_new    = scope1_ng_new + scope1_bio + scope1_oil
    return {
        'scope1_new_tCO2': round(scope1_new, 1),
    }

def scenario_ccs(scope1_residual_tCO2, capture_rate=0.90):
    assert 0.0 <= capture_rate <= 1.0, 'capture_rate must be between 0 and 1'
    residual = scope1_residual_tCO2 * (1 - capture_rate)
    return {
        'residual_tCO2': round(residual, 1),
    }

print("Scenario functions loaded.")

Scenario functions loaded.


In [19]:
# The deployment plan defines what levers are applied in each year.
# ppa:  fraction of electricity from renewable PPA (0.0 to 1.0)
# bio:  fraction of natural gas replaced by biomethane (0.0 to 1.0)
# ccs:  whether CCS is applied to residual Scope 1 (True or False)

LACQ_PLAN = {
    2024: {'ppa': 0.00, 'bio': 0.00, 'ccs': False},  # baseline year
    2026: {'ppa': 0.25, 'bio': 0.10, 'ccs': False},  # early steps
    2028: {'ppa': 0.50, 'bio': 0.20, 'ccs': False},
    2030: {'ppa': 1.00, 'bio': 0.40, 'ccs': False},  # EU Fit for 55
    2035: {'ppa': 1.00, 'bio': 0.60, 'ccs': True },
    2040: {'ppa': 1.00, 'bio': 0.70, 'ccs': True },
    2050: {'ppa': 1.00, 'bio': 0.80, 'ccs': True },  # net-zero ambition
}

print("Deployment plan defined.")
print(f"Years covered: {list(LACQ_PLAN.keys())}")

Deployment plan defined.
Years covered: [2024, 2026, 2028, 2030, 2035, 2040, 2050]


In [21]:
def build_trajectory(site, ef_scope1, ef_elec, ef_bio, deployment_plan):
    """
    Build annual emissions trajectory from a deployment plan.
    Returns a pandas DataFrame with one row per year.
    """
    # Calculate the baseline — what the site emits with no interventions
    ng_mwh          = site.get('natural_gas_MWh', 0)
    oil_mwh         = site.get('fuel_oil_MWh', 0)
    elec_mwh        = site['electricity_MWh']
    grid            = site['grid']

    scope1_base = (ng_mwh * ef_scope1['natural_gas'] +
                   oil_mwh * ef_scope1['fuel_oil'])
    scope2_base = elec_mwh * ef_elec[grid]
    baseline    = scope1_base + scope2_base

    # Build the trajectory row by row
    rows = []

    for year in sorted(deployment_plan.keys()):
        plan = deployment_plan[year]

        # Apply PPA to Scope 2
        ppa_result = scenario_ppa(site, ef_elec, plan.get('ppa', 0))
        scope2_y   = ppa_result['scope2_new_tCO2']

        # Apply biomethane to Scope 1
        bio_result = scenario_biomethane(site, ef_scope1, ef_bio,
                                         plan.get('bio', 0))
        scope1_y   = bio_result['scope1_new_tCO2']

        # Apply CCS to residual Scope 1 if flagged
        if plan.get('ccs', False):
            ccs_result = scenario_ccs(scope1_y)
            scope1_y   = ccs_result['residual_tCO2']

        total_y = scope1_y + scope2_y

        rows.append({
            'year':       year,
            'scope1':     round(scope1_y, 0),
            'scope2':     round(scope2_y, 0),
            'total':      round(total_y, 0),
            'reduction_pct': round((baseline - total_y) / baseline * 100, 1),
        })

    # Convert list of dictionaries to a pandas DataFrame
    df = pd.DataFrame(rows).set_index('year')
    return df, baseline

traj, baseline = build_trajectory(LACQ, SCOPE1, SCOPE2_ELECTRICITY,
                                  BIOMETHANE, LACQ_PLAN)
print(traj)
print(f"\nBaseline: {baseline:,.0f} tCO2eq/year")

        scope1   scope2     total  reduction_pct
year                                            
2024  197800.0  13750.0  211550.0            0.0
2026  181480.0  10938.0  192418.0            9.0
2028  165160.0   8125.0  173285.0           18.1
2030  132520.0   2500.0  135020.0           36.2
2035    9988.0   2500.0   12488.0           94.1
2040    8356.0   2500.0   10856.0           94.9
2050    6724.0   2500.0    9224.0           95.6

Baseline: 211,550 tCO2eq/year


In [23]:
# EU Fit for 55 requires -55% by 2030 vs 1990 baseline.
# I use 2024 as the reference year which is conservative.
target_2030 = baseline * 0.45   # 55% reduction means 45% remaining

print("Trajectory summary:")
print(traj.to_string())
print(f"\nBaseline:       {baseline:>12,.0f} tCO2eq/year")
print(f"2030 target:    {target_2030:>12,.0f} tCO2eq/year  (EU Fit for 55, -55%)")
print(f"2030 modelled:  {traj.loc[2030, 'total']:>12,.0f} tCO2eq/year")

if traj.loc[2030, 'total'] <= target_2030:
    print("2030 target MET under this deployment plan.")
else:
    gap = traj.loc[2030, 'total'] - target_2030
    print(f"2030 target MISSED by {gap:,.0f} tCO2eq. Plan needs strengthening.")

Trajectory summary:
        scope1   scope2     total  reduction_pct
year                                            
2024  197800.0  13750.0  211550.0            0.0
2026  181480.0  10938.0  192418.0            9.0
2028  165160.0   8125.0  173285.0           18.1
2030  132520.0   2500.0  135020.0           36.2
2035    9988.0   2500.0   12488.0           94.1
2040    8356.0   2500.0   10856.0           94.9
2050    6724.0   2500.0    9224.0           95.6

Baseline:            211,550 tCO2eq/year
2030 target:          95,198 tCO2eq/year  (EU Fit for 55, -55%)
2030 modelled:       135,020 tCO2eq/year
2030 target MISSED by 39,822 tCO2eq. Plan needs strengthening.
